# Optimising rating factors with `open-rating-engine`

This notebook takes the motor rating algorithm built in `01_rating_algorithm_example.ipynb` and re-tunes the multipliers in its `risk.demand_multiplier` rating tables, to maximise expected profit while holding the conversion rate where it is today.

| Step | What it does |
|---|---|
| Baseline | Loads the saved pipeline and scores a book of new-business quotes |
| Simulate | Re-scores the book across a grid of price multipliers, recording conversion and profit for every vehicle at every price point |
| Optimise | `PriceOptimizerRatingFactor` searches for one multiplier per rating-table row, reading each vehicle's response from the grid |
| Efficient frontier | Repeats the optimisation across a range of conversion targets, to show the profit/volume trade-off |
| Deploy | Writes the optimised multipliers back into the pipeline as a `RatingTablesLayer`, re-scores the book and saves the pipeline |

The optimiser never calls the pipeline itself. All the expensive scoring happens once, in the simulation step, and the optimiser interpolates between those results. The deploy step re-scores the book with the optimised tables, to check that interpolation against the real pipeline.

**Prerequisites**

- Run `01_rating_algorithm_example.ipynb` first. It saves the baseline pipeline to `models/rating_engines/pricing_pipeline.pkl`.
- Install the package with the ONNX, H2O and notebook extras. From the `open-rating-engine` folder: `pip install -e ".[onnx,h2o,notebook]"`
- Java 8+ installed, with `JAVA_HOME` set. The demand model is an H2O MOJO, scored in a JVM.
- Run the notebook from the `.examples` folder, so the `data/` and `models/` folders resolve.

In [ ]:
from copy import deepcopy
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mlflow.pyfunc import PythonModelContext

from open_rating_engine.optimization import PriceOptimizerRatingFactor, make_log_multiplier_regularizer

# Imported from the submodule: once it's loaded, the package attribute of the same
# name resolves to the module rather than the function.
from open_rating_engine.optimization.simulate_adjustment_grid import (
    make_demand_emulation,
    simulate_adjustment_grid,
)

## 1. Input data

The origination book: every policy is a new-business quote (`policy.transaction_type == "NB"`), so none of them has a previous premium and the renewal cap/cup in the pipeline never applies. Each vehicle's premium therefore moves in direct proportion to its demand multiplier.

In [ ]:
DATA_DIR = Path("data").resolve() / "origination"

data = {
    table: pd.read_parquet(DATA_DIR / f"{table}.parquet")
    for table in ["policy", "risk", "cover", "claims_history"]
}

pd.DataFrame({"rows": {table: len(df) for table, df in data.items()}})

In [ ]:
data["policy"]["policy.transaction_type"].value_counts()

## 2. Baseline rating engine

The pipeline was saved with `cloudpickle`, so it holds its layers, catalog and data structure, but not its loaded models. The ONNX and H2O layers reload their model files in `load_context`.

The saved pipeline also holds the context it was built with, whose artifact paths are absolute paths on the machine that ran notebook 01. Building a fresh context from this folder keeps the notebook portable.

In [ ]:
MODELS_DIR = Path("models").resolve()
BASELINE_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline.pkl"

with BASELINE_PATH.open("rb") as file:
    pricing_engine = cloudpickle.load(file)

artifacts = {
    "motor_risk_cost_onnx": MODELS_DIR / "motor_risk_cost_onnx" / "pipeline_onnx.onnx",
    "motor_demand_mojo": MODELS_DIR / "motor_demand_h2o" / "model.zip",
    "h2o_gen_model": MODELS_DIR / "motor_demand_h2o" / "h2o-genmodel.jar",
}
missing = [key for key, path in artifacts.items() if not path.exists()]
assert not missing, f"Model files not found for {missing}. Run this notebook from open-rating-engine/.examples."

context = PythonModelContext(
    artifacts={key: str(path) for key, path in artifacts.items()},
    model_config={"environment": "local", "java_options": "-Xmx1g"},
)
pricing_engine.load_context(context)

# The grid below scores the book many times over; one progress bar per score is noise.
pricing_engine.options["show_progress"] = False

print(f"{len(pricing_engine.get_layer_names())} layers loaded")

### The rating factors to optimise

`risk.demand_multiplier` is a `RatingTablesLayer` with two tables, youngest-driver age band and customer tenure band. Its formula multiplies them, so a vehicle's multiplier is its age-band value × its tenure-band value. Every value is 1.0 in the baseline, so these 9 rows are the decision variables.

In [ ]:
LAYER_NAME = "risk.demand_multiplier"

demand_layer = pricing_engine.layers[LAYER_NAME]
print(f"formula: {demand_layer.formula}")
print(f"default_values: {demand_layer.default_values}")

for name, table in demand_layer.rating_tables.items():
    print(f"\n{name}")
    display(table)

### Score the baseline

In [ ]:
baseline_output = pricing_engine.predict(data)

with pd.option_context("display.max_columns", None):
    display(baseline_output["risk"].head(10))

Book-level KPIs, as averages per vehicle. *Expected* figures are weighted by the probability that the customer accepts the quote, so expected profit is the profit the book makes once the quotes that don't convert are taken out.

In [ ]:
KPI_COLUMNS = {
    "Offered GWP": "risk.gwp",
    "Conversion rate": "risk.conversion_rate",
    "Expected GWP": "risk.reporting_expected_gwp",
    "Expected profit": "risk.reporting_expected_profit",
}


def book_kpis(scored):
    risk = scored["risk"]
    kpis = {label: risk[column].mean() for label, column in KPI_COLUMNS.items()}
    kpis["Expected profit margin"] = kpis["Expected profit"] / kpis["Expected GWP"]
    return pd.Series(kpis)


baseline_kpis = book_kpis(baseline_output)
baseline_kpis.to_frame("Baseline").round(4)

## 3. Simulate the price response

`simulate_adjustment_grid` scores the book once for each adjustment factor and returns an `(N, K)` array for every requested column: one row per vehicle, one column per factor.

How the factor reaches the pipeline is up to the `premium_emulation` callback. `make_demand_emulation` builds one that drives a single rating table of the demand layer. In `"override"` mode it sets every value in `risk_age_band`, and that table's default, to the factor. The tenure table is all 1.0, so the layer's output equals the factor on every vehicle, and each column of the grid is the book priced at `factor ×` its baseline premium.

**Choosing the grid.** The optimiser looks up each vehicle's response at its *effective* multiplier, the product of its age-band and tenure-band values. With each value bounded to `[0.9, 1.1]`, the product ranges over `[0.81, 1.21]`, so the grid runs from 0.80 to 1.25. Outside the grid, the interpolation holds the end values flat, so a grid narrower than the reachable range would hide the price response at the extremes.

Each factor is scored on a deep copy of the pipeline, in parallel threads.

In [ ]:
OUTPUT_COLUMNS = {
    "gwp": "risk.gwp",
    "conversion_rate": "risk.conversion_rate",
    "expected_gwp": "risk.reporting_expected_gwp",
    "expected_profit": "risk.reporting_expected_profit",
}

premium_emulation = make_demand_emulation(
    layer_name=LAYER_NAME,
    table_key="risk_age_band",
    mode="override",
)

simulated = simulate_adjustment_grid(
    data=data,
    pipeline=pricing_engine,
    premium_emulation=premium_emulation,
    output_table="risk",
    output_columns=OUTPUT_COLUMNS,
    adjustment_factor=np.round(np.arange(0.80, 1.2501, 0.05), 2),
    parallel_n_jobs=5,
)

factors = simulated["adjustment_factor_array"]
print(f"factors: {factors}")
print(f"conversion_rate array: {simulated['conversion_rate'].shape} (vehicles x factors)")

Averaging the grid over the book gives the response to a **uniform** price change: every vehicle's premium moved by the same factor. The 1.00 column reproduces the baseline KPIs.

In [ ]:
uniform_response = pd.DataFrame(
    {name: simulated[name].mean(axis=0) for name in OUTPUT_COLUMNS},
    index=pd.Index(factors, name="adjustment_factor"),
)


def at_factor(factor):
    # Book-average KPIs when every premium is scaled by `factor`.
    return uniform_response.loc[np.isclose(uniform_response.index, factor)].iloc[0]


current_conversion = at_factor(1.0)["conversion_rate"]
print(f"Current conversion: {current_conversion:.2%}")

uniform_response.round(4)

## 4. Optimise

### Objective and regularisation

The optimiser maximises the book-average of `objective_array`, here each vehicle's expected profit, and holds the book-average of `constraint_array` (conversion rate) at a target.

A `regularizer` is subtracted from the objective to keep the solution close to the current rates. `make_log_multiplier_regularizer` penalises `l1 · |log m| + l2 · (log m)²` for every multiplier `m`. The penalty is in the objective's units (dollars of expected profit per vehicle), and treats a 5% increase and the matching decrease alike. Both weights are 0 here, so the optimiser is unconstrained within its bounds. Raise them to trade some profit for smaller rate moves.

In [ ]:
regularizer = make_log_multiplier_regularizer(l1_penalty=0.0, l2_penalty=0.0)

### Build the optimiser from the pipeline

`from_pipeline` reads the decision space straight from the layer: each rating table becomes one rating factor, its rows become the categories, and the current values seed the search. It maps every vehicle in `scored_data` to its row in each table, and holds vehicles that match no row at the table's default.

In [ ]:
optimizer = PriceOptimizerRatingFactor.from_pipeline(
    pipeline=pricing_engine,
    layer_name=LAYER_NAME,
    scored_data=baseline_output,
    adjustment_factors=factors,
    objective_array=simulated["expected_profit"],
    constraint_array=simulated["conversion_rate"],
    reporting_arrays={
        "gwp": simulated["gwp"],
        "expected_gwp": simulated["expected_gwp"],
    },
    regularizer=regularizer,
)

### Maximise profit at today's conversion rate

`optimize` supports three `scipy` methods:

- `"SLSQP"` enforces the conversion target as an equality constraint.
- `"L-BFGS-B"` and `"Nelder-Mead"` add `constraint_penalty_weight × (conversion − target)²` to the objective instead.

Nelder-Mead is used here, with a large penalty weight to hold conversion close to the target.

In [ ]:
BOUNDS = (0.9, 1.1)

result = optimizer.optimize(
    target_constraint=float(current_conversion),
    bounds=BOUNDS,
    method="Nelder-Mead",
    constraint_penalty_weight=1e6,
)

result["summary"].T

`to_rating_tables` decodes the optimal multipliers back into the layer's table format, with the original key column and a `value` column.

In [ ]:
optimized_tables = optimizer.to_rating_tables()

for name, table in optimized_tables.items():
    exposure = baseline_output["risk"][table.columns[0]].value_counts().rename("vehicles")
    print(f"\n{name}")
    display(table.join(exposure, on=table.columns[0]).round(4))

## 5. Efficient frontier

Re-running the optimisation across a range of conversion targets traces the most expected profit available at each level of conversion. The range runs from the conversion of a uniform 15% price increase to that of a uniform 15% decrease. Further out, a target needs nearly every multiplier at its bound, which leaves the optimiser no room to move.

In [ ]:
target_range = np.linspace(at_factor(1.15)["conversion_rate"], at_factor(0.85)["conversion_rate"], 9)

frontier = optimizer.calculate_efficient_frontier(
    target_constraint_range=target_range,
    bounds=BOUNDS,
    method="SLSQP",
)

frontier[["target_constraint", "constraint", "objective", "price_multiplier", "gwp", "expected_gwp", "result_nit"]].round(4)

The chart compares the frontier with the uniform price change from section 3. A uniform change is always a feasible solution, so the frontier should sit on or above the uniform line. Wherever it falls below, the optimiser stopped short of the best solution for that target.

In [ ]:
UNIFORM_COLOR = "#eb6834"
FRONTIER_COLOR = "#2a78d6"
TEXT_COLOR = "#52514e"

optimized = result["summary"].iloc[0]

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    uniform_response["conversion_rate"],
    uniform_response["expected_profit"],
    color=UNIFORM_COLOR, linewidth=2, marker="o", markersize=5,
    label="Uniform price change",
)
ax.plot(
    frontier["constraint"],
    frontier["objective"],
    color=FRONTIER_COLOR, linewidth=2, marker="o", markersize=5,
    label="Optimised rating factors (frontier)",
)
ax.scatter(
    [current_conversion], [baseline_kpis["Expected profit"]],
    s=90, color="white", edgecolor=TEXT_COLOR, linewidth=2, zorder=3,
)
ax.annotate(
    "Baseline", (current_conversion, baseline_kpis["Expected profit"]),
    xytext=(-14, -18), textcoords="offset points", ha="right", color=TEXT_COLOR,
)
ax.scatter(
    [optimized["constraint"]], [optimized["objective"]],
    s=90, color=FRONTIER_COLOR, edgecolor="white", linewidth=2, zorder=4,
)
ax.annotate(
    "Optimised at current conversion", (optimized["constraint"], optimized["objective"]),
    xytext=(12, 8), textcoords="offset points", color=TEXT_COLOR,
)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"-${-y:,.0f}" if y < 0 else f"${y:,.0f}"))
ax.set_xlabel("Conversion rate", color=TEXT_COLOR)
ax.set_ylabel("Expected profit per vehicle", color=TEXT_COLOR)
ax.set_title("Expected profit vs conversion", loc="left")
ax.grid(color="#e4e3df", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(colors=TEXT_COLOR)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

## 6. Deploy the optimised rating factors

`to_layer` rebuilds the `RatingTablesLayer` from the optimised tables, keeping the original layer's name, formula, defaults, dependencies and output column. Swapping it into a copy of the pipeline leaves the baseline untouched for comparison.

In [ ]:
optimized_engine = deepcopy(pricing_engine)
optimized_engine.remove_parameter(LAYER_NAME)
optimized_engine.add_parameter(optimizer.to_layer())

for name, table in optimized_engine.layers[LAYER_NAME].rating_tables.items():
    print(f"\n{name}")
    display(table.round(4))

### Re-score the book

Scoring with the real pipeline checks the optimiser's interpolated estimate. The two agree closely because the premium is proportional to the multiplier on new business, and the grid is fine enough for linear interpolation between its points to track the demand model.

In [ ]:
optimized_output = optimized_engine.predict(data)

with pd.option_context("display.max_columns", None):
    display(optimized_output["risk"].head(10))

In [ ]:
optimized_kpis = book_kpis(optimized_output)

comparison = pd.DataFrame({
    "Baseline": baseline_kpis,
    "Optimised (re-scored)": optimized_kpis,
    "Optimiser estimate": {
        "Offered GWP": optimized["gwp"],
        "Conversion rate": optimized["constraint"],
        "Expected GWP": optimized["expected_gwp"],
        "Expected profit": optimized["objective"],
        "Expected profit margin": optimized["objective"] / optimized["expected_gwp"],
    },
})
comparison["Change vs baseline"] = comparison["Optimised (re-scored)"] / comparison["Baseline"] - 1
comparison.round(4)

In [ ]:
print(f"Conversion:      {baseline_kpis['Conversion rate']:.2%} -> {optimized_kpis['Conversion rate']:.2%}")
print(f"Expected profit: ${baseline_kpis['Expected profit']:.2f} -> ${optimized_kpis['Expected profit']:.2f} per vehicle "
      f"({optimized_kpis['Expected profit'] / baseline_kpis['Expected profit'] - 1:+.2%})")
print(f"Profit margin:   {baseline_kpis['Expected profit margin']:.2%} -> {optimized_kpis['Expected profit margin']:.2%}")

### Results by rating factor

Book KPIs split by each rating factor show where the optimiser moved price, and how volume and profit shifted between the bands.

In [ ]:
def kpis_by(scored, column, order):
    risk = scored["risk"]
    return risk.groupby(column, observed=True).agg(
        vehicles=("risk.risk_number", "count"),
        demand_multiplier=("risk.demand_multiplier", "mean"),
        conversion_rate=("risk.conversion_rate", "mean"),
        expected_profit=("risk.reporting_expected_profit", "mean"),
    ).reindex(order)


for name, table in optimized_tables.items():
    column = table.columns[0]
    print(f"\n{name}")
    display(
        pd.concat(
            {
                "Baseline": kpis_by(baseline_output, column, table[column]),
                "Optimised": kpis_by(optimized_output, column, table[column]),
            },
            axis=1,
        ).round(4)
    )

## 7. Save the optimised pipeline

Saved with `cloudpickle` beside the baseline, in the same way as notebook 01. Load it with `cloudpickle.load` and call `load_context` before scoring.

In [ ]:
OPTIMIZED_PATH = MODELS_DIR / "rating_engines" / "pricing_pipeline_rating_factor_optimized.pkl"

with OPTIMIZED_PATH.open("wb") as file:
    cloudpickle.dump(optimized_engine, file)

print(f"Pipeline saved to {OPTIMIZED_PATH}") 